# Day 8: Intro to PyTorch & TensorFlow — Module 2 Capstone

**Module 2 — Neural Network Basics | 100 Days of Data Science**

## Why This Matters
Days 4-7 built every piece of a neural network by hand: perceptrons, MLPs, activations, forward/backward propagation, loss functions, and optimizers. Today we put it all together the way it's actually done in practice — using a real framework, on a real dataset, end to end.

This is the **Module 2 capstone**: a complete PyTorch training pipeline, plus the equivalent in TensorFlow/Keras so you can read either.

## Topics Covered Today
1. PyTorch core building blocks recap (tensors, autograd, nn.Module)
2. Loading a real dataset (`sklearn` digits — handwritten digit images)
3. Full PyTorch training pipeline: Dataset, DataLoader, model, training loop, evaluation
4. The same pipeline in TensorFlow/Keras
5. PyTorch vs TensorFlow — when to use which
6. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

print("NumPy version:", np.__version__)

---
## 1. PyTorch Core Building Blocks (Recap)

| Concept | What it is | Where you've seen it |
|---|---|---|
| `torch.tensor` | N-dimensional array, like a NumPy array with GPU + gradient support | Day 1 (tensors) |
| `requires_grad=True` | Tells PyTorch to track operations for gradient computation | Day 2 (autograd) |
| `.backward()` | Computes all gradients automatically via the chain rule | Day 2, Day 6 |
| `nn.Module` | Base class for defining a model's layers and forward pass | Day 4 |
| `nn.Linear`, `nn.ReLU`, etc. | Pre-built layers and activations | Day 4, Day 5 |
| `nn.CrossEntropyLoss`, `nn.MSELoss` | Pre-built loss functions | Day 3, Day 7 |
| `torch.optim.Adam`, `.SGD` | Pre-built optimizers | Day 7 |

---
## 2. Loading a Real Dataset

The `sklearn` digits dataset: 1,797 images of handwritten digits (0-9), each 8x8 pixels. Small enough to train quickly, real enough to be a genuine classification task.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target  # X: (1797, 64) flattened 8x8 images, y: (1797,) labels 0-9

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))

# Visualize a few samples
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f"Label: {y[i]}")
    ax.axis('off')
plt.show()

# Normalize pixel values to [0, 1] and split into train/test
X = X / 16.0  # pixel values range 0-16 in this dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("\nTrain size:", X_train.shape[0], "| Test size:", X_test.shape[0])

---
## 3. Full PyTorch Training Pipeline

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    torch.manual_seed(42)

    # --- Step 1: Dataset & DataLoader ---
    class DigitsDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.long)

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    train_dataset = DigitsDataset(X_train, y_train)
    test_dataset = DigitsDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    print("PyTorch, NumPy, sklearn all loaded successfully.")
    print("Number of training batches:", len(train_loader))
except ImportError:
    print("PyTorch not installed. Run: pip install torch")

In [ ]:
# --- Step 2: Model definition (everything from Days 4-5 combined) ---
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 10)  # 10 classes, raw logits (softmax applied inside the loss)
        )

    def forward(self, x):
        return self.net(x)

model = DigitClassifier()
print(model)

total_params = sum(p.numel() for p in model.parameters())
print("\nTotal trainable parameters:", total_params)

In [ ]:
# --- Step 3: Loss & optimizer (everything from Day 3 and Day 7 combined) ---
criterion = nn.CrossEntropyLoss()  # combines softmax + cross-entropy internally
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# --- Step 4: Training loop (everything from Day 6 combined) ---
epochs = 30
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {avg_loss:.4f}")

In [ ]:
# --- Step 5: Evaluation ---
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f} ({correct}/{total} correct)")

plt.figure(figsize=(6,4))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss Over Time')
plt.grid(True)
plt.show()

---
## 4. The Same Pipeline in TensorFlow/Keras

Same dataset, same architecture, same training process — different syntax. Keras is more declarative (define the whole model + `compile` + `fit`), while PyTorch is more explicit (you write the training loop yourself).

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.random.set_seed(42)

    # Model definition
    tf_model = keras.Sequential([
        layers.Input(shape=(64,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(10)  # raw logits, same as PyTorch version
    ])

    # Loss & optimizer (combined into .compile, unlike PyTorch's explicit loop)
    tf_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # Training loop -- .fit() replaces the manual for-loop
    history = tf_model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=30,
        batch_size=32,
        verbose=0
    )

    test_loss, test_acc = tf_model.evaluate(X_test, y_test, verbose=0)
    print(f"TensorFlow test accuracy: {test_acc:.4f}")

    plt.figure(figsize=(6,4))
    plt.plot(history.history['loss'], label='train loss')
    plt.plot(history.history['val_loss'], label='val loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('TensorFlow/Keras Training Curve')
    plt.legend()
    plt.grid(True)
    plt.show()
except ImportError:
    print("TensorFlow not installed. Run: pip install tensorflow")

---
## 5. PyTorch vs TensorFlow — When to Use Which

| Aspect | PyTorch | TensorFlow/Keras |
|---|---|---|
| Training loop | Explicit (you write it) | Implicit (`.fit()`) |
| Debugging | Easier — standard Python control flow | Historically harder (improved a lot with TF2 eager mode) |
| Research community | Dominant in research papers, Hugging Face ecosystem | Common in production/mobile (TF Lite, TF Serving) |
| Learning curve | Slightly steeper (you build more manually) | Gentler for beginners (high-level API) |
| Deployment | TorchServe, ONNX export | TF Serving, TF Lite, very mature |

**For this roadmap:** we'll primarily use **PyTorch** going forward (Modules 3-8), since it's the dominant framework in modern research, Hugging Face, and most current DL job postings — but now you can read TensorFlow code too if you encounter it.

---
## 6. Practice Exercises
Try these to wrap up Module 2:

1. Change the PyTorch model architecture (add a third hidden layer, or change neuron counts) and see if accuracy improves.
2. Swap `Adam` for plain `SGD` (`torch.optim.SGD(model.parameters(), lr=0.01)`) — how much slower is convergence?
3. Add a confusion matrix (`sklearn.metrics.confusion_matrix`) to see which digits get misclassified most often.
4. In the TensorFlow version, change `batch_size` to 8 and to 128 — how does training speed and final accuracy change?
5. Reflect: pick 3 concepts from Days 1-7 that you directly recognized being used inside this pipeline. Where exactly did they show up?

In [ ]:
# Your practice code here


---
## Summary
- Everything from Module 1 and Days 4-7 comes together in a real, working training pipeline
- **PyTorch**: `Dataset` → `DataLoader` → `nn.Module` → explicit training loop → evaluation
- **TensorFlow/Keras**: `Sequential` model → `.compile()` → `.fit()` → `.evaluate()`
- Both frameworks implement the exact same underlying math you built by hand on Day 6
- We'll use **PyTorch** as the primary framework for the rest of this roadmap

**Module 2 — Neural Network Basics is now complete!** (Days 4-8: Perceptron→MLP, Activations, Forward/Backward Prop, Loss/Optimizers, Framework Pipeline)

Next up: **Day 9 — Start of Module 3: Convolutional Neural Networks (Convolutions, Pooling, Padding)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*